In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath(".."))
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(os.path.abspath("."))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATASET_DIR = (PROJECT_ROOT / "data" / "research_processed_smoke_auto").resolve()
MODEL_DIR = (PROJECT_ROOT / "models" / "research_smoke_auto").resolve()
RESULTS_DIR = (PROJECT_ROOT / "results" / "research_smoke_auto").resolve()

# 04 GPT Adjudication

This workflow delegates the **final semantic decision layer** to GPT.

* GPT *only* processes anomaly candidates (not all windows) to reduce cost and noise.
* If the GPT label == `normal`, it marks the incident as `skipped_for_alerting`.

In [ ]:
import pandas as pd
from src.config import GPTConfig
from src.gpt_adjudicator import adjudicate_anomaly_records, build_window_summary, call_openai_responses_api
from src.utils import read_json

### Extract Candidates

In [ ]:
cfg = GPTConfig(
    evaluation_dir=RESULTS_DIR,
    output_dir=RESULTS_DIR,
    max_records=20, # Evaluates max top 20 verified candidates
)

try:
    confirmed_alerts = pd.read_csv(RESULTS_DIR / "realtime_alert_candidates.csv")
except FileNotFoundError:
    # Fallback if realtime alerts missing, get from streaming enriched.
    stream = pd.read_csv(RESULTS_DIR / "hybrid" / "realtime_stream_predictions_enhanced.csv")
    confirmed_alerts = stream[stream['is_candidate'] == True].copy()

print(f"Loaded {len(confirmed_alerts)} anomaly candidates for GPT processing.")

### Adjudication Flow

In [ ]:
adjudication_summary = adjudicate_anomaly_records(
    prediction_csv_path=RESULTS_DIR / "hybrid" / "realtime_stream_predictions.csv", # Update logic path
    config=cfg,
    max_records=cfg.max_records,
)

### Results Preview Table
Skip standard routing if GPT believes the state is 'normal'.

In [ ]:
try:
    gpt_results = pd.read_csv(RESULTS_DIR / "gpt_adjudicated_alerts.csv")
    # Apply logic: if GPT label == normal, mark as skipped
    is_normal = gpt_results['gpt_label'].str.lower().str.contains('normal', na=False)
    gpt_results['skipped_for_alerting'] = is_normal

    # Save the updated frame
    gpt_results.to_csv(RESULTS_DIR / "gpt_adjudicated_alerts.csv", index=False)

    display_cols = ['window_id', 'gpt_label', 'skipped_for_alerting', 'gpt_reason']
    display_cols = [c for c in display_cols if c in gpt_results.columns]

    print("GPT Decisions Preview:")
    display(gpt_results[display_cols].head(10))
except FileNotFoundError:
    print("No GPT records generated. Check API Key or candidate limits.")